# 05 - Evaluacion de resultados

Este notebook realiza una evaluacion practica del sistema de recomendacion desarrollado. El objetivo no es obtener una medida industrial avanzada, sino comprobar si las recomendaciones son coherentes y utiles desde un punto de vista sencillo y defendible para el TFG.

## 1. Como se va a evaluar el sistema

La evaluacion que se presenta aqui es una evaluacion practica basada en la coherencia de resultados. No se aplican metricas avanzadas propias de entornos industriales ni se dispone de una validacion completa con usuarios reales.

En su lugar, se seleccionan varias peliculas de prueba de distintos generos y se revisa si las recomendaciones comparten rasgos razonables con la pelicula original. Ademas, se calculan algunas metricas simples para resumir la calidad media de las recomendaciones obtenidas.

## 2. Importacion de librerias

In [12]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

base_dir = Path('..').resolve()
sys.path.append(str(base_dir))

from src.data_utils import load_tags
from src.evaluation import (
    build_evaluation_summary,
    calculate_recommendation_summary,
    compare_genres,
)
from src.recommender import (
    build_genre_feature_matrix,
    find_movie_index,
    get_genre_columns,
    recommend_movies_by_genres,
)

## 3. Carga de datos

In [13]:
movies_path = base_dir / 'data' / 'processed' / 'movies_clean.csv'
tags_path = base_dir / 'data' / 'raw' / 'tags.csv'
results_dir = base_dir / 'reports' / 'resultados'

movies_clean = pd.read_csv(movies_path)
movies_clean.head()

,movieId,title,genres,year,title_clean,rating_mean,rating_count,Action,Adventure,Animation,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1995.0,Toy Story,3.897438,68997,0,1,1,...,0,0,0,0,0,0,0,0,0,0
1,2,Jumanji (1995),Adventure|Children|Fantasy,1995.0,Jumanji,3.275758,28904,0,1,0,...,0,0,0,0,0,0,0,0,0,0
2,3,Grumpier Old Men (1995),Comedy|Romance,1995.0,Grumpier Old Men,3.139447,13134,0,0,0,...,0,0,0,0,0,1,0,0,0,0
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,1995.0,Waiting to Exhale,2.845331,2806,0,0,0,...,0,0,0,0,0,1,0,0,0,0
4,5,Father of the Bride Part II (1995),Comedy,1995.0,Father of the Bride Part II,3.059602,13154,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## 4. Preparacion del recomendador basado en generos

Primero cargamos la matriz de caracteristicas basada en generos, que sera la base de la evaluacion principal.

In [14]:
genre_columns = get_genre_columns(movies_clean)
feature_matrix = build_genre_feature_matrix(movies_clean, genre_columns)

print('Numero de peliculas:', len(movies_clean))
print('Numero de columnas de genero:', len(genre_columns))

Numero de peliculas: 87585
Numero de columnas de genero: 19


## 5. Seleccion de peliculas de prueba

Se eligen varias peliculas de distintos generos para comprobar el comportamiento del recomendador en casos variados.

In [15]:
test_movies = {
    'animacion': 'Toy Story',
    'accion': 'La la land',
    'comedia': 'Cinema Paradiso',
    'drama': 'Interstellar',
    'ciencia_ficcion': 'Matrix, The',
}

test_movies

{'animacion': 'Toy Story',
 'accion': 'La la land',
 'comedia': 'Cinema Paradiso',
 'drama': 'Interstellar',
 'ciencia_ficcion': 'Matrix, The'}

## 6. Ejecucion del recomendador y tablas de resultados

Para cada pelicula de prueba se obtienen recomendaciones, se comparan sus generos con los de la pelicula original y se muestran las tablas resultantes.

In [16]:
recommendation_examples = []
evaluation_results = []

for category, movie_title in test_movies.items():
    print(f'\n### Categoria: {category} | Pelicula: {movie_title}')
    recommendations = recommend_movies_by_genres(
        movies_clean,
        feature_matrix,
        movie_title,
        n=5,
        min_ratings=20,
    )

    movie_index = find_movie_index(movies_clean, movie_title)
    original_genres = movies_clean.loc[movie_index, 'genres'] if movie_index is not None else ''

    if not recommendations.empty:
        genre_comparison = recommendations['genres'].apply(
            lambda value: compare_genres(original_genres, value)
        )
        recommendations['shared_genres'] = genre_comparison.apply(lambda value: ', '.join(value['shared_genres']))
        recommendations['shared_genres_count'] = genre_comparison.apply(lambda value: value['shared_genres_count'])

        recommendations['input_movie'] = movie_title
        recommendations['category'] = category
        recommendation_examples.append(recommendations.copy())

        summary = calculate_recommendation_summary(recommendations)
        overlap_ratio = (recommendations['shared_genres_count'] > 0).mean()
        if summary['similarity_mean'] >= 0.8 and overlap_ratio >= 0.8:
            comment = 'Resultados muy coherentes con la pelicula original.'
        elif summary['similarity_mean'] >= 0.6:
            comment = 'Resultados razonables, aunque algo generales.'
        else:
            comment = 'Resultados mas debiles o demasiado amplios.'

        evaluation_results.append({
            'input_movie': movie_title,
            'category': category,
            'num_recommendations': summary['num_recommendations'],
            'similarity_mean': summary['similarity_mean'],
            'rating_mean_mean': summary['rating_mean_mean'],
            'rating_count_mean': summary['rating_count_mean'],
            'comment': comment,
        })

    display(recommendations)



### Categoria: animacion | Pelicula: Toy Story
Pelicula seleccionada: Toy Story (1995)


,title,year,genres,rating_mean,rating_count,similarity_score,shared_genres,shared_genres_count,input_movie,category
0,"Monsters, Inc.",2001.0,Adventure|Animation|Children|Comedy|Fantasy,3.837442,46036,1.0,"Adventure, Animation, Children, Comedy, Fantasy",5,Toy Story,animacion
1,Toy Story 2,1999.0,Adventure|Animation|Children|Comedy|Fantasy,3.812043,32683,1.0,"Adventure, Animation, Children, Comedy, Fantasy",5,Toy Story,animacion
2,Antz,1998.0,Adventure|Animation|Children|Comedy|Fantasy,3.213386,12857,1.0,"Adventure, Animation, Children, Comedy, Fantasy",5,Toy Story,animacion
3,"Emperor's New Groove, The",2000.0,Adventure|Animation|Children|Comedy|Fantasy,3.650783,11437,1.0,"Adventure, Animation, Children, Comedy, Fantasy",5,Toy Story,animacion
4,Moana,2016.0,Adventure|Animation|Children|Comedy|Fantasy,3.816381,8278,1.0,"Adventure, Animation, Children, Comedy, Fantasy",5,Toy Story,animacion



### Categoria: accion | Pelicula: La la land
Pelicula seleccionada: La La Land (2016)


,title,year,genres,rating_mean,rating_count,similarity_score,shared_genres,shared_genres_count,input_movie,category
0,Sleepless in Seattle,1993.0,Comedy|Drama|Romance,3.510240,32129,1.0,"Comedy, Drama, Romance",3,La la land,accion
1,Shakespeare in Love,1998.0,Comedy|Drama|Romance,3.773842,27514,1.0,"Comedy, Drama, Romance",3,La la land,accion
2,As Good as It Gets,1997.0,Comedy|Drama|Romance,3.771600,27176,1.0,"Comedy, Drama, Romance",3,La la land,accion
3,Lost in Translation,2003.0,Comedy|Drama|Romance,3.793686,26273,1.0,"Comedy, Drama, Romance",3,La la land,accion
4,Juno,2007.0,Comedy|Drama|Romance,3.704740,24094,1.0,"Comedy, Drama, Romance",3,La la land,accion



### Categoria: comedia | Pelicula: Cinema Paradiso
Pelicula seleccionada: Cinema Paradiso (Nuovo cinema Paradiso) (1989)


,title,year,genres,rating_mean,rating_count,similarity_score,shared_genres,shared_genres_count,input_movie,category
0,One Flew Over the Cuckoo's Nest,1975.0,Drama,4.204229,44592,1.0,Drama,1,Cinema Paradiso,comedia
1,Rain Man,1988.0,Drama,3.901394,33284,1.0,Drama,1,Cinema Paradiso,comedia
2,Cast Away,2000.0,Drama,3.683696,32200,1.0,Drama,1,Cinema Paradiso,comedia
3,Dead Poets Society,1989.0,Drama,3.938424,31035,1.0,Drama,1,Cinema Paradiso,comedia
4,Requiem for a Dream,2000.0,Drama,3.983923,29235,1.0,Drama,1,Cinema Paradiso,comedia



### Categoria: drama | Pelicula: Interstellar
Pelicula seleccionada: Interstellar (2014)


,title,year,genres,rating_mean,rating_count,similarity_score,shared_genres,shared_genres_count,input_movie,category
0,Edge of Tomorrow,2014.0,Action|Sci-Fi|IMAX,3.958198,19760,0.816497,"IMAX, Sci-Fi",2,Interstellar,drama
1,Gravity,2013.0,Action|Sci-Fi|IMAX,3.597484,18126,0.816497,"IMAX, Sci-Fi",2,Interstellar,drama
2,Cloud Atlas,2012.0,Drama|Sci-Fi|IMAX,3.560508,8065,0.816497,"IMAX, Sci-Fi",2,Interstellar,drama
3,The Amazing Spider-Man 2,2014.0,Action|Sci-Fi|IMAX,3.012546,4304,0.816497,"IMAX, Sci-Fi",2,Interstellar,drama
4,Contagion,2011.0,Sci-Fi|Thriller|IMAX,3.504482,4239,0.816497,"IMAX, Sci-Fi",2,Interstellar,drama



### Categoria: ciencia_ficcion | Pelicula: Matrix, The
Pelicula seleccionada: Matrix, The (1999)


,title,year,genres,rating_mean,rating_count,similarity_score,shared_genres,shared_genres_count,input_movie,category
0,"Terminator, The",1984.0,Action|Sci-Fi|Thriller,3.902092,47131,1.0,"Action, Sci-Fi, Thriller",3,"Matrix, The",ciencia_ficcion
1,Blade Runner,1982.0,Action|Sci-Fi|Thriller,4.110005,46289,1.0,"Action, Sci-Fi, Thriller",3,"Matrix, The",ciencia_ficcion
2,Predator,1987.0,Action|Sci-Fi|Thriller,3.672675,20776,1.0,"Action, Sci-Fi, Thriller",3,"Matrix, The",ciencia_ficcion
3,X-Men: The Last Stand,2006.0,Action|Sci-Fi|Thriller,3.208563,13991,1.0,"Action, Sci-Fi, Thriller",3,"Matrix, The",ciencia_ficcion
4,Johnny Mnemonic,1995.0,Action|Sci-Fi|Thriller,2.762982,13788,1.0,"Action, Sci-Fi, Thriller",3,"Matrix, The",ciencia_ficcion


## 7. Tabla resumen de la evaluacion

A partir de los resultados anteriores se construye una tabla resumen con informacion cuantitativa y un comentario breve para cada pelicula de entrada.

In [17]:
evaluation_summary = build_evaluation_summary(evaluation_results)
display(evaluation_summary)

,input_movie,category,num_recommendations,similarity_mean,rating_mean_mean,rating_count_mean,comment
0,Toy Story,animacion,5,1.000000,3.666007,22258.2,Resultados muy coherentes con la pelicula orig...
1,La la land,accion,5,1.000000,3.710822,27437.2,Resultados muy coherentes con la pelicula orig...
2,Cinema Paradiso,comedia,5,1.000000,3.942333,34069.2,Resultados muy coherentes con la pelicula orig...
3,Interstellar,drama,5,0.816497,3.526644,10898.8,Resultados muy coherentes con la pelicula orig...
4,"Matrix, The",ciencia_ficcion,5,1.000000,3.531263,28395.0,Resultados muy coherentes con la pelicula orig...


## 8. Casos donde el sistema funciona bien

En general, el recomendador funciona mejor cuando la pelicula de entrada tiene una combinacion de generos clara y reconocible. En esos casos, las recomendaciones suelen mantener una estructura similar y una similitud coseno elevada.

In [18]:
best_cases = evaluation_summary.sort_values('similarity_mean', ascending=False).head(3)
display(best_cases[['input_movie', 'similarity_mean', 'comment']])

,input_movie,similarity_mean,comment
1,La la land,1.0,Resultados muy coherentes con la pelicula orig...
4,"Matrix, The",1.0,Resultados muy coherentes con la pelicula orig...
2,Cinema Paradiso,1.0,Resultados muy coherentes con la pelicula orig...


## 9. Limitaciones observadas

El sistema tambien presenta limites que conviene dejar claros para la memoria:

- Algunas recomendaciones pueden ser demasiado generales cuando muchas peliculas comparten exactamente los mismos generos.
- El sistema depende mucho de los generos, por lo que no distingue bien matices de tono, estilo o tematica.
- Existe un sesgo hacia peliculas populares cuando se priorizan titulos con mas valoraciones.
- Si no se usan tags o sinopsis, falta informacion semantica que podria enriquecer la recomendacion.

In [19]:
worst_cases = evaluation_summary.sort_values('similarity_mean', ascending=True).head(3)
display(worst_cases[['input_movie', 'similarity_mean', 'comment']])

,input_movie,similarity_mean,comment
3,Interstellar,0.816497,Resultados muy coherentes con la pelicula orig...
0,Toy Story,1.000000,Resultados muy coherentes con la pelicula orig...
2,Cinema Paradiso,1.000000,Resultados muy coherentes con la pelicula orig...


## 10. Comparacion opcional con la version de generos + tags

Si existe `tags.csv`, se puede construir una version enriquecida del recomendador combinando generos y tags. Esta comparacion no sustituye la evaluacion principal, pero ayuda a ver si los tags aportan informacion mas especifica.

In [22]:
if tags_path.exists():
    tags_df = load_tags(tags_path)
    tags_clean = tags_df[['movieId', 'tag']].copy()
    tags_clean['tag'] = tags_clean['tag'].fillna('').str.lower().str.strip()
    tags_clean = tags_clean[tags_clean['tag'] != '']

    tags_grouped = (
        tags_clean.groupby('movieId')['tag']
        .apply(lambda values: ' '.join(values))
        .reset_index(name='tags_combined')
    )

    movies_with_tags = movies_clean.merge(tags_grouped, on='movieId', how='left')
    movies_with_tags['tags_combined'] = movies_with_tags['tags_combined'].fillna('')
    movies_with_tags['genres_text'] = movies_with_tags['genres'].fillna('').str.replace('|', ' ', regex=False)
    movies_with_tags['content_features'] = (
        movies_with_tags['genres_text'].str.strip() + ' ' + movies_with_tags['tags_combined'].str.strip()
    ).str.strip()

    vectorizer = TfidfVectorizer(stop_words='english', min_df=2)
    content_matrix = vectorizer.fit_transform(movies_with_tags['content_features'])

    def recommend_movies_by_content(movie_title, n=5, min_ratings=20):
        movie_index = find_movie_index(movies_with_tags, movie_title)
        if movie_index is None:
            return pd.DataFrame(columns=['title', 'year', 'genres', 'rating_mean', 'rating_count', 'similarity_score'])

        similarity_scores = cosine_similarity(content_matrix[movie_index], content_matrix).flatten()
        candidate_indices = [idx for idx in similarity_scores.argsort()[::-1] if idx != movie_index]

        recommendations = movies_with_tags.loc[candidate_indices, ['title_clean', 'title', 'year', 'genres', 'rating_mean', 'rating_count']].copy()
        recommendations['title'] = recommendations['title_clean'].fillna(recommendations['title'])
        recommendations['similarity_score'] = similarity_scores[candidate_indices]
        recommendations = recommendations[['title', 'year', 'genres', 'rating_mean', 'rating_count', 'similarity_score']]
        recommendations = recommendations.sort_values(['similarity_score', 'rating_count', 'rating_mean'], ascending=[False, False, False])

        if min_ratings > 0:
            high_confidence = recommendations[recommendations['rating_count'] >= min_ratings]
            fallback = recommendations[recommendations['rating_count'] < min_ratings]
            recommendations = pd.concat([high_confidence, fallback], ignore_index=True)

        return recommendations.head(n).reset_index(drop=True)

    comparison_movie = 'Interstellar'
    print('Recomendador solo con generos')
    display(recommend_movies_by_genres(movies_clean, feature_matrix, comparison_movie, n=5, min_ratings=20))

    print('Recomendador con generos y tags')
    display(recommend_movies_by_content(comparison_movie, n=5, min_ratings=20))
else:
    print('No se ha encontrado tags.csv, por lo que no se realiza la comparacion opcional.')

Recomendador solo con generos
Pelicula seleccionada: Interstellar (2014)


,title,year,genres,rating_mean,rating_count,similarity_score
0,Edge of Tomorrow,2014.0,Action|Sci-Fi|IMAX,3.958198,19760,0.816497
1,Gravity,2013.0,Action|Sci-Fi|IMAX,3.597484,18126,0.816497
2,Cloud Atlas,2012.0,Drama|Sci-Fi|IMAX,3.560508,8065,0.816497
3,The Amazing Spider-Man 2,2014.0,Action|Sci-Fi|IMAX,3.012546,4304,0.816497
4,Contagion,2011.0,Sci-Fi|Thriller|IMAX,3.504482,4239,0.816497


Recomendador con generos y tags


,title,year,genres,rating_mean,rating_count,similarity_score
0,2010: The Year We Make Contact,1984.0,Sci-Fi,3.394928,4673,0.491453
1,Breach,2020.0,Action|Sci-Fi,1.684211,38,0.491430
2,Space Cop,2016.0,Action|Comedy|Sci-Fi,2.560976,41,0.476272
3,Contact,1997.0,Drama|Sci-Fi,3.701131,25896,0.452221
4,"Spirit of '76, The",1990.0,Comedy|Sci-Fi,2.916667,42,0.450453


## 11. Guardado de resultados

Por ultimo, se guardan algunos ejemplos de recomendaciones y la tabla resumen de evaluacion para reutilizarlos en la memoria o en otros informes.

In [21]:
results_dir.mkdir(parents=True, exist_ok=True)

recommendations_examples_df = pd.concat(recommendation_examples, ignore_index=True) if recommendation_examples else pd.DataFrame()
recommendations_examples_df.to_csv(results_dir / 'recommendations_examples.csv', index=False)
evaluation_summary.to_csv(results_dir / 'evaluation_summary.csv', index=False)

print('Archivos guardados en:', results_dir)

Archivos guardados en: C:\Users\alexo\Desktop\TFG\reports\resultados


## 12. Cierre

La evaluacion realizada es sencilla, pero permite comprobar de forma razonable si el sistema genera recomendaciones coherentes. Ademas, ayuda a identificar fortalezas y limites del modelo antes de plantear mejoras posteriores.